# Part 5: Trend Analysis and Insight Generation

**Objective:** Analyze the structured outputs generated by the LLM to identify patterns, trends, and key insights from airline customer feedback.

**Description:** In this final stage, the structured information produced by the LLM is analyzed to uncover patterns in customer complaints and emerging trends. The enriched dataset, which now includes fields such as complaint topics, severity levets, root causes, and trend signals, is used to perform exploratory analysis. 

A series of visualizations are generated to explore relationships between different variables, such as complaint topics, severity levels, and detected trend signals. These visual analyses help highlight recurring customer issues and potential emerging problems discussed by users. 

The results of this analysis provide actionable insights that can support data-driven business decisions, such as identifying critical service issues, monitoring customer sentiment, and detecting early signals of operational problems. 

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# read parquet
data = pd.read_parquet("tweets_enriched.parquet")
data.head(5)

# rename the LLM output
data = data.rename(columns={"0": "llm_output"})

In [ ]:
# explore the formats and the output
n_cluster = sorted(data["cluster"].unique())

for c in n_cluster: 
    print(c)
    print(data["llm_output"][data["cluster"] == c].value_counts())

In [ ]:
# clean your json column 
data["llm_output"] = (data["llm_output"]
                      .str.replace("```json", "", regex=False)
                      .str.replace("```", "", regex=False)
                      .str.strip()
                    )
data["llm_output"][0]

In [ ]:
# add your json into columns for the dataset. 
data["llm_output"] = data["llm_output"].apply(json.loads) # convert string JSON to dictionary
data_expanded = pd.concat(
    [
        data.drop(columns=["llm_output"]),
        pd.json_normalize(data["llm_output"]) # Turn the dictionary column into columns
    ],
    axis = 1
)

### Let's get into our analysis: 

In [ ]:
# Obtain the top complaint_topics
data_expanded.groupby("complaint_topic").size().sort_values(ascending=False)

# Make a barplot 
counts = data_expanded["complaint_topic"].value_counts()

sns.barplot(
    x=counts.index,
    y=counts.values,
    palette="pastel"
)

plt.xticks(rotation=45)
plt.ylabel("Count")
plt.xlabel("Complaint topic")
plt.title("Complaint topic Distribution")
plt.show()

# Análisis 

In [ ]:
# Check which issues are within a complaint topic 
pivot = pd.crosstab(data_expanded["complaint_topic"], data_expanded["customer_issue"])

plt.figure(figsize=(10,6))
sns.heatmap(
    pivot,
    annot=True,
    fmt="d",
    cmap="magma"
)

plt.xlabel("Customer Issue")
plt.ylabel("Complaint topic")
plt.title("Distribution of complaint topics vs Customer Issues")
plt.show()

# Análisis 

In [ ]:
# customer_issue vs. likely root cause.
pivot = pd.crosstab(data_expanded["customer_issue"], data_expanded["likely_root_cause"])

plt.figure(figsize=(15,6))
sns.heatmap(
    pivot,
    annot=True,
    fmt="d",
    cmap="magma"
)

plt.xlabel("Likely root cause")
plt.xticks(rotation=75)
plt.ylabel("Customer Issue")
plt.title("Distribution of Customer issues vs. likely root cause")
plt.show()

In [ ]:
# Top topics per airline
pivot = pd.crosstab(data_expanded["airline"], data_expanded["complaint_topic"])

plt.figure(figsize=(10,6))
sns.heatmap(
    pivot,
    annot=True,
    fmt="d",
    cmap="magma"
)

plt.xlabel("Complaint topic")
plt.ylabel("Airline")
plt.title("Distribution of complaint topics vs Customer Issues")
plt.show()

In [ ]:
# Severity vs complaint_topic
pivot = pd.crosstab(data_expanded["severity"], data_expanded["complaint_topic"])

plt.figure(figsize=(15,6))
sns.heatmap(
    pivot,
    annot=True,
    fmt="d",
    cmap="magma"
)

plt.xlabel("Complaint topic")
plt.xticks(rotation=45)
plt.ylabel("Severity")
plt.title("Distribution of Complaint topic vs severity")
plt.show()

In [ ]:
# customer issues vs severity
pivot = pd.crosstab(data_expanded["severity"], data_expanded["customer_issue"])

plt.figure(figsize=(15,6))
sns.heatmap(
    pivot,
    annot=True,
    fmt="d",
    cmap="magma"
)

plt.xlabel("Likely root cause")
plt.xticks(rotation=75)
plt.ylabel("Customer Issue")
plt.title("Distribution of Customer issues vs. likely root cause")
plt.show()

In [ ]:
# Complaint_topic vs trend_signal
pivot = data_expanded.pivot_table(
    index="complaint_topic",
    columns="trend_signal",
    aggfunc="size",
    fill_value=0
)

pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(10,6),
    colormap="viridis"
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Complaint topic")
plt.ylabel("Count")
plt.title("Trend Signals per Complaint Topic")

plt.tight_layout()

plt.show()

In [ ]:
# trend_signal vs customer issue
pivot = data_expanded.pivot_table(
    index="customer_issue",
    columns="trend_signal",
    aggfunc="size",
    fill_value=0
)

pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(10,6),
    colormap="viridis"
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Complaint topic")
plt.ylabel("Count")
plt.title("Trend Signals per Customer issue")

plt.tight_layout()

plt.show()

In [ ]:
# negativereason vs complaint topic to see improvement on llm output
pd.crosstab(data_expanded["negativereason"], data_expanded["complaint_topic"])

In [ ]:
location_columns = ['location_atlanta', 'location_austin',
       'location_boston', 'location_brooklyn', 'location_california',
       'location_chicago', 'location_columbus', 'location_ct',
       'location_dallas', 'location_denver', 'location_houston',
       'location_las vegas', 'location_london', 'location_los angeles',
       'location_miami', 'location_nashville', 'location_new york',
       'location_new york city', 'location_ny', 'location_nyc',
       'location_other', 'location_philadelphia', 'location_portland',
       'location_san diego', 'location_san francisco', 'location_seattle',
       'location_texas', 'location_toronto', 'location_usa',
       'location_washington', 'location_washington dc']

data_expanded["location"] = (
    data_expanded[location_columns]
    .idxmax(axis=1)
    .str.replace("location_","", regex=False)
)

data.head(5)

pivot = data_expanded.pivot_table(
    index = "location",
    columns = "complaint_topic",
    aggfunc = "size",
    fill_value = 0
)

pivot

In [ ]:
te falta ver vs el sentimiento de los comentarios
y si puedes te falta ver igual los comentarios, en qué puntos tienes más comentarios y de qué tipo 
y además, falta checar como han evolucionado los comentarios, es decir si han ido reduciendo o no. 
